## Cleaning Each Dataset

In [2]:
# install libraries
# %pip install pandas openpyxl numpy matplotlib seabor

import pandas as pd

In [3]:
costs = pd.read_excel("../data/raw/costs.xlsx")

In [4]:
gates = pd.read_excel("../data/raw/gates.xlsx")

In [5]:
transactions = pd.read_excel("../data/raw/transactions.xlsx")

## Costs

In [6]:
costs.head(10)

,Cost_ID,Cost_Date,Cost_Type,Route_ID,From,To,Amount
0,CO000001,2024-09-01 06:30:00,Crew,YVR-YLW,Vancouver,Kelowna,1388.53
1,CO000002,2024-09-01 06:30:00,Food,yvr-ylw,Vancouver,Kelowna,170.52
2,CO000003,2024-09-01 06:30:00,Fuel,YVR-YLW,Vancouver,Kelowna,1401.75
3,CO000004,2024-09-01 06:30:00,Maintenance,yvrylw,Vancouver,Kelowna,1657.45
4,CO000005,2024-09-01 06:35:00,Crew,YLW-YYC,Kelowna,Calgary,1673.28
5,CO000006,2024-09-01 06:35:00,Food,ylwyyc,Kelowna,Calgary,221.60
6,CO000007,2024-09-01 06:35:00,Fuel,YLW-YYC,Kelowna,Calgary,1800.44
7,CO000008,2024-09-01 06:35:00,Maintenance,YLW-YYC,Kelowna,Calgary,1898.61
8,CO000009,2024-09-01 06:45:00,Crew,YYZYYC,Toronto,Calgary,12105.73
9,CO000010,2024-09-01 06:45:00,Food,YYZYYC,Toronto,Calgary,2037.11


In [7]:
print(costs.isna().sum())
print(costs.columns.tolist())
# costs.info()
# costs.shape
# costs.duplicated().sum()

Cost_ID      0
Cost_Date    0
Cost_Type    0
Route_ID     0
From         0
To           0
Amount       0
dtype: int64
['Cost_ID', 'Cost_Date', 'Cost_Type', 'Route_ID', 'From', 'To', 'Amount']


In [8]:
#Standardizing Costs Data


# change route_id to UPPER-UPPER (FROM-TO)
costs["Route_ID"] = (costs["Route_ID"].str.upper().str.replace("-", "", regex=False).str.replace(r"^([A-Z]{3})([A-Z]{3})$", r"\1-\2", regex=True)
)

In [9]:
costs.head(20)

,Cost_ID,Cost_Date,Cost_Type,Route_ID,From,To,Amount
0,CO000001,2024-09-01 06:30:00,Crew,YVR-YLW,Vancouver,Kelowna,1388.53
1,CO000002,2024-09-01 06:30:00,Food,YVR-YLW,Vancouver,Kelowna,170.52
2,CO000003,2024-09-01 06:30:00,Fuel,YVR-YLW,Vancouver,Kelowna,1401.75
3,CO000004,2024-09-01 06:30:00,Maintenance,YVR-YLW,Vancouver,Kelowna,1657.45
4,CO000005,2024-09-01 06:35:00,Crew,YLW-YYC,Kelowna,Calgary,1673.28
5,CO000006,2024-09-01 06:35:00,Food,YLW-YYC,Kelowna,Calgary,221.60
6,CO000007,2024-09-01 06:35:00,Fuel,YLW-YYC,Kelowna,Calgary,1800.44
7,CO000008,2024-09-01 06:35:00,Maintenance,YLW-YYC,Kelowna,Calgary,1898.61
8,CO000009,2024-09-01 06:45:00,Crew,YYZ-YYC,Toronto,Calgary,12105.73
9,CO000010,2024-09-01 06:45:00,Food,YYZ-YYC,Toronto,Calgary,2037.11


In [10]:
costs.shape[0] / 4

28801.0

In [11]:
flight_costs_df = costs.pivot(
    index=["Cost_Date", "Route_ID", "From", "To"],
    columns="Cost_Type",
    values="Amount"
).reset_index()

flight_costs_df.columns.name = None

In [12]:
flight_costs_df["Total"] = flight_costs_df["Crew"] + flight_costs_df["Food"] + flight_costs_df["Fuel"] + flight_costs_df["Maintenance"]


In [13]:
flight_costs_df.sort_values(by = "Total", ascending = False)

,Cost_Date,Route_ID,From,To,Crew,Food,Fuel,Maintenance,Total
19881,2026-02-10 07:15:00,YUL-YVR,Montreal,Vancouver,25489.05,2991.50,26516.10,30007.23,85003.88
19901,2026-02-10 16:30:00,YUL-YVR,Montreal,Vancouver,24131.35,2940.79,26978.56,28802.79,82853.49
18807,2026-01-13 16:30:00,YUL-YVR,Montreal,Vancouver,25194.77,2370.44,26288.76,28577.51,82431.48
18787,2026-01-13 07:15:00,YUL-YVR,Montreal,Vancouver,25560.29,2532.20,25164.07,28584.92,81841.48
20008,2026-02-13 07:35:00,YYZ-YVR,Toronto,Vancouver,26364.18,3425.44,25303.17,25761.87,80854.66
...,...,...,...,...,...,...,...,...,...
1000,2024-10-06 17:50:00,YVR-YYJ,Vancouver,Victoria,1022.16,123.65,1107.77,1066.22,3319.80
15458,2025-10-26 17:50:00,YVR-YYJ,Vancouver,Victoria,1040.41,106.01,1091.33,1057.65,3295.40
626,2024-09-23 07:35:00,YVR-YYJ,Vancouver,Victoria,979.47,154.21,1003.72,1091.46,3228.86
1060,2024-10-08 17:50:00,YVR-YYJ,Vancouver,Victoria,935.15,122.74,1059.30,979.84,3097.03


In [14]:
# save flight costs df
flight_costs_df.to_csv("../data/processed/flight_costs_df.csv", index=False)

## Gates

In [15]:
gates.head(10)

,City,Gate_Number,Route_ID,Year,Annual_Gate_Cost
0,Abbotsford,G1,yxx-yyc,2024-2025,166800
1,Calgary,B1,YYCYVR,2024-2025,373200
2,Calgary,B1,yycyyz,2024-2025,373200
3,Calgary,B1,YYCYEG,2024-2025,373200
4,Calgary,B2,yyc-ywg,2024-2025,351900
5,Calgary,B2,YYCYXE,2024-2025,351900
6,Calgary,B2,YYC-YQR,2024-2025,351900
7,Calgary,B3,YYC-YLW,2024-2025,340700
8,Calgary,B3,yycyxx,2024-2025,340700
9,Edmonton,B1,yeg-yvr,2024-2025,258800


In [16]:
# gates.info()
# gates.shape
print(gates.isna().sum())
# gates.duplicated().sum()
gates[gates["Route_ID"].isna()]

City                0
Gate_Number         0
Route_ID            4
Year                0
Annual_Gate_Cost    0
dtype: int64


,City,Gate_Number,Route_ID,Year,Annual_Gate_Cost
16,Halifax,A3,NaN,2024-2025,220700
28,Ottawa,A2,NaN,2024-2025,205000
72,Halifax,A3,NaN,2025-2026,225200
84,Ottawa,A2,NaN,2025-2026,208300


In [17]:
# Standardizing Gates Data

# change Route_ID to UPPER-UPPER (FROM-TO)
gates["Route_ID"] = (gates["Route_ID"].str.upper().str.replace("-", "", regex=False).str.replace(r"^([A-Z]{3})([A-Z]{3})$", r"\1-\2", regex=True))

In [18]:
gates[gates["Gate_Number"] == "A3"]
gates[gates["Gate_Number"] == "A2"]

,City,Gate_Number,Route_ID,Year,Annual_Gate_Cost
14,Halifax,A2,YHZ-YYT,2024-2025,231900
15,Halifax,A2,YHZ-YQM,2024-2025,231900
28,Ottawa,A2,NaN,2024-2025,205000
30,Quebec City,A2,YQB-YUL,2024-2025,171300
31,Regina,A2,YQR-YYC,2024-2025,161200
33,St. John's,A2,YYT-YYZ,2024-2025,184800
34,St. John's,A2,YYT-YHZ,2024-2025,184800
45,Vancouver,A2,YVR-YYC,2024-2025,455100
46,Vancouver,A2,YVR-YEG,2024-2025,455100
55,Yellowknife,A2,YZF-YEG,2024-2025,150000


In [19]:
# saving gates
gates.to_csv("../data/processed/gates_clean.csv", index=False)


## Transactions

In [20]:
transactions.head(10)

,Transaction_ID,Booking_ID,Transaction_Date,Route_ID,Transaction_Type,Membership_Tier,Booking_Channel,Payment_Method,Amount
0,TX1525667,BK396819,2026-07-23 18:18,YYZYHZ,Ticket,Non-Member,Direct Web,Credit Card,954.80
1,TX1420893,BK245942,2026-03-14 05:23,yvr-yeg,Ticket,Blue,Corporate Portal,Credit Card,971.17
2,TX1313706,BK274024,2025-10-29 11:54,yvryeg,Food & Onboard,Non-Member,Online Travel Agency,Credit Card,9.87
3,TX1169582,BK374084,2025-04-30 02:50,YYT-YHZ,Ticket,Silver,Direct Web,Credit Card,744.61
4,TX1378465,BK319834,01/20/2026 04:02,YYZYYC,Ticket,Blue,Direct Web,Credit Card,1380.38
5,TX1133037,BK382994,2025-03-11 20:20,YQBYUL,Extra Baggage,Blue,Direct Web,Credit Card,45.06
6,TX1112418,BK332533,2025-02-12 10:58,YHZ-YYZ,Change Fee,Blue,Direct Web,Points + Cash,75.00
7,TX1363471,BK329720,2026-01-01 01:29,YWG-YYC,TICKET,Non-Member,Corporate Portal,Credit Card,1231.75
8,TX1413482,BK440357,2026-03-04 18:04,YYC-YVR,Ticket,Blue,Mobile App,Credit Card,837.35
9,TX1056794,BK321270,2024-11-28 02:15,YLW-YVR,Ticket,Non-Member,Mobile App,Debit,599.02


In [21]:
# transactions.info()
# transactions.shape
transactions.isna().sum()
# transactions["Transaction_ID"].duplicated().sum()
# transactions["Transaction_ID"].duplicated(keep = False).sum()
# transactions.duplicated().sum()

Transaction_ID         0
Booking_ID             0
Transaction_Date       0
Route_ID               0
Transaction_Type       0
Membership_Tier     4105
Booking_Channel        0
Payment_Method      4456
Amount                 0
dtype: int64

In [22]:
transactions

,Transaction_ID,Booking_ID,Transaction_Date,Route_ID,Transaction_Type,Membership_Tier,Booking_Channel,Payment_Method,Amount
0,TX1525667,BK396819,2026-07-23 18:18,YYZYHZ,Ticket,Non-Member,Direct Web,Credit Card,954.80
1,TX1420893,BK245942,2026-03-14 05:23,yvr-yeg,Ticket,Blue,Corporate Portal,Credit Card,971.17
2,TX1313706,BK274024,2025-10-29 11:54,yvryeg,Food & Onboard,Non-Member,Online Travel Agency,Credit Card,9.87
3,TX1169582,BK374084,2025-04-30 02:50,YYT-YHZ,Ticket,Silver,Direct Web,Credit Card,744.61
4,TX1378465,BK319834,01/20/2026 04:02,YYZYYC,Ticket,Blue,Direct Web,Credit Card,1380.38
...,...,...,...,...,...,...,...,...,...
559980,TX1057608,BK116129,2024-11-29 04:55,yow-yyz,Ticket,Non-Member,Direct Web,Credit Card,519.17
559981,TX1344855,BK279337,2025-12-08 08:47,YUL-YOW,Ticket,Non-Member,Direct Web,Credit Card,613.72
559982,TX1102189,BK246017,2025-01-30 00:47,YLWYVR,Ticket,Non-Member,DIRECT WEB,Credit Card,557.37
559983,TX1359241,BK345688,2025-12-26 12:36,YHZYUL,Ticket,Silver,Mobile App,Credit Card,867.80


In [23]:
transactions[transactions["Payment_Method"].isna()].head(10)

,Transaction_ID,Booking_ID,Transaction_Date,Route_ID,Transaction_Type,Membership_Tier,Booking_Channel,Payment_Method,Amount
97,TX1442024,BK267958,2026-04-10 09:23,YYZYYT,Ticket,Non-Member,Online Travel Agency,NaN,1284.57
139,TX1491286,BK281263,2026-06-13 04:07,yhz-yqm,TICKET,NONE,OTA,NaN,466.16
323,TX1371365,BK317404,2026-01-11 01:23,YXU-YYZ,Ticket,Non-Member,Direct Web,NaN,435.66
389,TX1245750,BK270221,2025-08-02 05:02,YVR-YLW,TICKET,Non-Member,Direct Web,NaN,678.37
408,TX1498962,BK177931,2026-06-22 05:08,yytyyz,Ticket,Non-Member,Mobile App,NaN,1244.24
468,TX1477610,BK370008,2026-05-27 13:20,YVR-YUL,Ticket,Gold,Online Travel Agency,NaN,1749.98
595,TX1203725,BK130611,2025-06-12 20:43,YYTYHZ,Ticket,Non-Member,Online Travel Agency,NaN,1005.79
634,TX1533057,BK376107,2026-08-01 11:45,YVR-YYC,Food & Onboard,Blue,Mobile App,NaN,22.47
881,TX1450033,BK372448,2026-04-21 06:54,yxuyyz,Ticket,Gold,Mobile App,NaN,528.45
925,TX1092191,BK404528,2025-01-16 08:10,YEG-YVR,Ticket,Non-Member,Mobile App,NaN,713.89


919 rows are exactly the same, 1856 rows have the same transaction ID

In [24]:
transactions[transactions["Transaction_ID"].duplicated(keep = False)].sort_values(by = "Transaction_ID").head(10)

,Transaction_ID,Booking_ID,Transaction_Date,Route_ID,Transaction_Type,Membership_Tier,Booking_Channel,Payment_Method,Amount
391057,TX1000138,BK188785,2024-09-01 05:39,yxeyyc,Ticket,Non-Member,Mobile App,points + cash,708.58
50092,TX1000138,BK188785,2024-09-01 05:39,yxeyyc,Ticket,Non-Member,Mobile App,POINTS + CASH,708.58
107840,TX1000200,BK247854,09/01/2024 08:06,YHZYUL,Ticket,Non-Member,Corporate Portal,Points + Cash,1019.00
18883,TX1000200,BK247854,09/01/2024 08:06,YHZYUL,Ticket,Non-Member,Corporate Portal,Points + Cash,1019.00
243826,TX1001267,BK146372,2024-09-03 10:44,yhz-yyz,Ticket,Non-Member,Website,Travel Voucher,1256.40
442476,TX1001267,BK146372,2024-09-03 10:44,yhz-yyz,Ticket,Non-Member,Website,Travel Voucher,1256.40
28478,TX1002597,BK418396,2024-09-06 01:19,YYC-YYZ,Ticket,Blue,Direct Web,Credit Card,1402.03
301935,TX1002597,BK418396,2024-09-06 01:19,YYC-YYZ,Ticket,Blue,Direct Web,credit card,1402.03
4546,TX1003061,BK194363,2024-09-06 18:31,ylw-yvr,Ticket,Non-Member,Mobile App,Credit Card,490.43
56702,TX1003061,BK194363,2024-09-06 18:31,ylw-yvr,Ticket,Non-Member,Mobile App,Credit Card,490.43


In [25]:
# Checking uniformity of categorial data
print(transactions["Membership_Tier"].unique())
print(transactions["Payment_Method"].unique())


<StringArray>
[  'Non-Member',         'Blue',       'Silver',     'Platinum',
        'Gold ',         'Gold',         'NONE',  ' Non-Member',
      'Silver ',            nan,  'Non-Member ',        'Blue ',
        ' Blue', ' Non-Member ',    ' Platinum',       ' Gold ',
       ' Blue ',      ' Silver',     ' Silver ',    'Platinum ',
   ' Platinum ',        ' Gold',      'Unknown']
Length: 23, dtype: str
<StringArray>
[     'Credit Card',    'Points + Cash',            'Debit',
    ' Credit Card ',           'Debit ',            'debit',
     'Credit Card ',   'Travel Voucher',      'credit card',
   'travel voucher',    ' credit card ',                nan,
     ' Credit Card',      'CREDIT CARD',    'points + cash',
 ' Travel Voucher ',            'DEBIT',          ' Debit ',
   'Points + Cash ',    'POINTS + CASH',    ' CREDIT CARD ',
  ' Points + Cash ',   'TRAVEL VOUCHER',           ' Debit',
   ' Points + Cash',  ' Travel Voucher',     'credit card ',
           ' debit',     '

In [26]:
# Standarizing Transaction_Date
transactions['Transaction_Date'] = pd.to_datetime(transactions['Transaction_Date'], format = 'mixed')

# Standardizing Route_ID
transactions['Route_ID'] = (
    transactions['Route_ID']
    .str.upper()
    .str.replace(" ", "-")
    .str.replace(r"^([A-Z]{3})([A-Z]{3})$", r"\1-\2", regex=True)
)

# Standardizing Booking_Channel
transactions["Booking_Channel"] = (
    transactions["Booking_Channel"]
    .str.strip()
    .str.title()
)

channel_map = {
    # Direct Web
    "Web": "Direct Web",
    
    # Mobile App
    "App": "Mobile App",
    "Mobile": "Mobile App",
    
    # Corporate Portal
    "Corp Portal": "Corporate Portal",
    "Corporate": "Corporate Portal",
    
    # Online Travel Agency
    "Online Travel Agcy": "Online Travel Agency",
    "ota": "Online Travel Agency",
    "OTA": "Online Travel Agency",
    "Ota": "Online Travel Agency"
}
transactions["Booking_Channel"] = transactions["Booking_Channel"].replace(channel_map)

# Standardizing Payment_Method
transactions["Payment_Method"] = (
    transactions["Payment_Method"]
    .str.strip()
    .str.title()
)

# Standardizing Transaction_Type
transactions["Transaction_Type"] = (
    transactions["Transaction_Type"]
    .str.strip()
    .str.title()
)

# Fill in NA values in Membership_Tier with "Non-Member"
transactions["Membership_Tier"] = transactions["Membership_Tier"].fillna("Non-Member")
transactions.head(10)

,Transaction_ID,Booking_ID,Transaction_Date,Route_ID,Transaction_Type,Membership_Tier,Booking_Channel,Payment_Method,Amount
0,TX1525667,BK396819,2026-07-23 18:18:00,YYZ-YHZ,Ticket,Non-Member,Direct Web,Credit Card,954.80
1,TX1420893,BK245942,2026-03-14 05:23:00,YVR-YEG,Ticket,Blue,Corporate Portal,Credit Card,971.17
2,TX1313706,BK274024,2025-10-29 11:54:00,YVR-YEG,Food & Onboard,Non-Member,Online Travel Agency,Credit Card,9.87
3,TX1169582,BK374084,2025-04-30 02:50:00,YYT-YHZ,Ticket,Silver,Direct Web,Credit Card,744.61
4,TX1378465,BK319834,2026-01-20 04:02:00,YYZ-YYC,Ticket,Blue,Direct Web,Credit Card,1380.38
5,TX1133037,BK382994,2025-03-11 20:20:00,YQB-YUL,Extra Baggage,Blue,Direct Web,Credit Card,45.06
6,TX1112418,BK332533,2025-02-12 10:58:00,YHZ-YYZ,Change Fee,Blue,Direct Web,Points + Cash,75.00
7,TX1363471,BK329720,2026-01-01 01:29:00,YWG-YYC,Ticket,Non-Member,Corporate Portal,Credit Card,1231.75
8,TX1413482,BK440357,2026-03-04 18:04:00,YYC-YVR,Ticket,Blue,Mobile App,Credit Card,837.35
9,TX1056794,BK321270,2024-11-28 02:15:00,YLW-YVR,Ticket,Non-Member,Mobile App,Debit,599.02


In [27]:
transactions["Transaction_Type"].unique()

<StringArray>
[        'Ticket', 'Food & Onboard',  'Extra Baggage',     'Change Fee',
         'Refund',   'Seat Upgrade']
Length: 6, dtype: str

In [28]:
transactions["Booking_ID"].duplicated().sum()


np.int64(206983)

In [29]:
transactions[transactions["Booking_ID"].duplicated(keep = False)].sort_values(by = "Booking_ID").head(10)

,Transaction_ID,Booking_ID,Transaction_Date,Route_ID,Transaction_Type,Membership_Tier,Booking_Channel,Payment_Method,Amount
368256,TX1050966,BK100001,2024-11-19 19:38:00,YLW-YVR,Food & Onboard,Non-Member,Corporate Portal,Credit Card,27.81
512273,TX1047114,BK100001,2024-11-14 08:59:00,YLW-YVR,Ticket,Non-Member,Corporate Portal,Credit Card,558.37
304812,TX1009702,BK100006,2024-09-18 19:35:00,YWG-YYC,Ticket,Non-Member,Mobile App,Credit Card,824.53
202061,TX1009701,BK100006,2024-09-18 19:35:00,YYC-YVR,Ticket,Non-Member,Mobile App,Credit Card,514.77
247362,TX1536249,BK100010,2026-08-05 10:58:00,YVR-YYC,Ticket,Blue,Website,Credit Card,834.79
39607,TX1536250,BK100010,2026-08-05 10:58:00,YYC-YWG,Ticket,Blue,Direct Web,Credit Card,1107.79
268912,TX1048828,BK100011,2024-11-16 18:02:00,YYZ-YWG,Food & Onboard,Non-Member,Mobile App,Credit Card,28.08
530677,TX1024220,BK100011,2024-10-12 04:54:00,YYZ-YWG,Ticket,Non-Member,Mobile App,Credit Card,1364.45
520116,TX1474899,BK100012,2026-05-24 04:23:00,YEG-YZF,Ticket,Non-Member,Direct Web,Credit Card,1051.87
479518,TX1484726,BK100012,2026-06-05 09:34:00,YEG-YZF,Seat Upgrade,Non-Member,Direct Web,Credit Card,58.08


In [30]:
transactions["Transaction_Type"].unique()

<StringArray>
[        'Ticket', 'Food & Onboard',  'Extra Baggage',     'Change Fee',
         'Refund',   'Seat Upgrade']
Length: 6, dtype: str

In [31]:
# row per booking id

booking_df = transactions.pivot_table(
    index="Booking_ID",
    columns="Transaction_Type",
    values="Amount",
    aggfunc="sum",
    fill_value=0
).reset_index()

booking_df.columns.name = None

booking_df["Total"] = booking_df[
    ["Ticket", "Food & Onboard", "Extra Baggage",
     "Change Fee", "Seat Upgrade", "Refund"]
].sum(axis=1)

In [32]:
booking_df["Total"] = booking_df["Change Fee"] + booking_df["Extra Baggage"] +booking_df["Food & Onboard"] + + booking_df["Ticket"]+ booking_df["Seat Upgrade"] - booking_df["Refund"]

In [33]:
booking_df.duplicated().sum()

np.int64(0)

In [34]:
# row per booking route
# separates booking route based on rout id (connecting flights)
booking_route_df = transactions.pivot_table(
    index=["Booking_ID", "Route_ID"],
    columns="Transaction_Type",
    values="Amount",
    aggfunc="sum",
    fill_value=0
).reset_index()

booking_route_df.columns.name = None

In [35]:
booking_route_df["Total"] = booking_route_df["Change Fee"] + booking_route_df["Ticket"] + booking_route_df["Extra Baggage"] + booking_route_df["Food & Onboard"] + booking_route_df["Seat Upgrade"] - booking_route_df["Refund"]

In [36]:
booking_route_df.sort_values(by = "Booking_ID").duplicated().sum()

np.int64(0)

In [37]:
booking_df.isna().sum()

Booking_ID        0
Change Fee        0
Extra Baggage     0
Food & Onboard    0
Refund            0
Seat Upgrade      0
Ticket            0
Total             0
dtype: int64

In [38]:
# save transaction data
booking_df.to_csv("../data/processed/bookings.csv", index=False)
booking_route_df.to_csv("../data/processed/booking_routes.csv", index=False)

In [39]:
booking_df.head(10)

,Booking_ID,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total
0,BK100000,0.0,0.0,0.00,0.0,0.0,683.89,683.89
1,BK100001,0.0,0.0,27.81,0.0,0.0,558.37,586.18
2,BK100002,0.0,0.0,0.00,0.0,0.0,796.89,796.89
3,BK100003,0.0,0.0,0.00,0.0,0.0,596.12,596.12
4,BK100004,0.0,0.0,0.00,0.0,0.0,743.32,743.32
5,BK100005,0.0,0.0,0.00,0.0,0.0,1395.40,1395.40
6,BK100006,0.0,0.0,0.00,0.0,0.0,1339.30,1339.30
7,BK100007,0.0,0.0,0.00,0.0,0.0,865.09,865.09
8,BK100008,0.0,0.0,0.00,0.0,0.0,562.55,562.55
9,BK100009,0.0,0.0,0.00,0.0,0.0,934.58,934.58


In [40]:
booking_route_df["Change Fee"].max()

np.float64(200.0)